# Gridworld y su solución como MDPs

En este trabajo definiremos el ambiente de Gridworld y su solución como un MDP.
Gridworld es un ambiente clásico de prueba dentro del aprendizaje por refuerzo. Durante este taller definiremos el modelo básico del ambiente, que extenderemos incrementalmente de acuerdo a las necesidades del algoritmo de solución.

## Ambiente 🌎

El ambiente de gridworld se define como una cuadricula de `nxm`. El ambiente tiene obstaculos, es decir casillas por las cuales no puede pasar el agente. Al chocar con un obstaculo, el agente terminaría en el mismo estado inicial. Además, el ambiente tiene una casilla de inicio, y algunas casillas de salida. Un ejemplo del ambiente para el caso `3x4` se muestra a continuación.

![gridworld.png](./img/gridworld.png)

En este ejemplo del ambiente el agente comienza en la casilla inferior izquierda y tiene como objetivo llegar a la casilla de salida verde, con recompensa 1. La otra casilla de salida, tiene recompensa -1.




### Task 1.
#### ¿Cómo podemos codificar el ambiente?

De una definición completa del ambiente, como una clase de python llamada `Environment`, estableciendo:
1. Un atributo que define la cuadrícula (`board`). El ambiente recibirá una matriz como parámetro describiendo la cuadrícula en el momento de su creación. Definiremos las casillas por las que puede pasar el agente como casillas vacias, las casillas por las que no puede pasar el agente con un valor none `#` y las casillas de salida con el valor asociado a la recompensa definidas para cada una de ellas.
2. Un atributo `nrows` para almacenar la cantidad de filas de la cuadrícula.
3. Un atributo `ncols` para almacenar la cantidad de columnas de la cuadrícula.
4. Un atributo `initial_state` para almacenar el estado inicial del agente dentro del ambiente.
5. Un atributo con el estado actual (`current_state`) en el que se encuentra el agente. El valor de `current_state` se definirá como una tupla 
6. Un atributo `P` que guarda la matriz de probabilidades de cada una de las acciones para cada estado. Dicha matriz esta definida por parámetro.

Un ejemplo de la definición del tablero para el caso de 5x5 de la figura anterior se da a continuación.
```
board = [['', ' ', ' ',  '+1'],
         [' ', '#', ' ',  '-1'],
         ['S', ' ', ' ', ' ']]
```
En el ejemplo `S` denota el estado inicial y `'#'` la casilla prohibida (manejaremos esta convención para todos los ambientes de gridworld).

De forma similar a la definición del `board` la matriz de probabilidades `P` se define como:
```
P = [[[0.15, 0.15, 0, 0.8], [0.15, 0.15, 0, 0.8], [0.15, 0.15, 0, 0.8],  [1]],
         [[0.8, 0, 0.15, 0.15], '#', [0.8, 0, 0.15, 0.15],  [-1]],
         [[0.8, 0, 0.15, 0.15], [0.15, 0.15, 0.8, 0], [0.15, 0.15, 0.8, 0], [0.15, 0.15, 0.8, 0]]]
```
Para la definición de `P` vamos a entender cada una de las posiciones de la probabilidad en el orden (`'up', 'down', 'left', 'right'`), es decir, para la casilla superior izquierda la probabilidad de tomar la acción `up` y llegar a la casilla de arriba es de `0.15`, el agente llegara a cualquiera de las otras casillas con probabilidad `0.85` (equiprobables entre ellas). En las casillas de salida, el agente solo tiene una posibilidad que es tomar la acción `exit` que le da la recompensa asociada a la casilla al agente.


#### Comportamiento del ambeinte

Una vez definido el ambiente definimos su comportamiento. Para ello requerimos los siguientes métodos:
1. `get_current_state` que no recibe parámetros y retorna el estado actual (la casilla donde se encuentra el agente)
2. `get_posible_actions` que recibe el estado actual del agente como parámetro y retorna las acciones disponibles para dicho estado. Las acciones estarán dadas por su nombre (`'up', 'down', 'left', 'right'`) para las casillas normales y (`'exit'`) para las casillas de salida. Como convención definiremos que el agente siempre puede moverse en todas las direcciones, donde un movimiento en dirección de un obstáculo o los límites del ambiente no tienen ningún efecto visible en la posición del agente.
3. `do_action` que recibe como parámetro la acción a ejecutar y retorna el valor de la recompensa y el nuevo estado del agente, como un pareja `reward, new_state`. Note que `do_action` esta restringida por la matriz de probabilidad `P` para la ejecución real de las acciones.
4. `reset` que no recibe parámetros y restablece el ambiente a su estado inicial.
5. `is_terminal` que no recibe parámetros y determina si el agente está en el estado final o no. En nuestro caso, el estado final estará determinado por las casillas de salida (i.e., con un valor definido).



In [66]:
import random

class Environment:
    def __init__(self, board, P):
        self.board = board
        self.P = P
        self.nrows = len(board)
        self.ncols = len(board[0]) 
        self.initial_state = self._find_initial_state()
        self.current_state = self.initial_state

    def _find_initial_state(self):
        for i in range(self.nrows):
            for j in range(self.ncols):
                if self.board[i][j] == 'S':
                    return (i, j)
        return (0, 0)

    def get_current_state(self):
        return self.current_state

    def get_posible_actions(self, state):
        r, c = state
        if self._is_exit(r, c):
            return ['exit']
        return ['up', 'down', 'left', 'right']

    def do_action(self, idx_action):
        r, c = self.current_state

        # Si estamos en estado terminal
        if self._is_exit(r, c):
            if idx_action == 4:  # 'exit' action
                return float(self.board[r][c]), self.current_state
            return 0, self.current_state


        probs = self.P[r][c][idx_action]
        if probs ==0:
            return 0, self.current_state

        actions = ['up', 'down', 'left', 'right', 'exit']

        # Elegimos la acción real según probabilidades
        chosen_action = random.choices(actions, weights=self.P[r][c], k=1)[0]
        nr, nc = self._move(r, c, chosen_action)
        # print(f"Chosen action: {chosen_action}, New position: ({nr}, {nc})")
        self.current_state = (nr, nc)

        if self._is_exit(nr, nc):
            reward = float(self.board[nr][nc])
        else:
            reward = 0
        return reward, self.current_state


    def reset(self):
        self.current_state = self.initial_state

    def is_terminal(self):
        r, c = self.current_state
        return self._is_exit(r, c)

    def _is_exit(self, r, c):
        val = self.board[r][c]
        return isinstance(val, str) and val.strip() in ['+1', '-1'] or isinstance(val, (int, float))

    def _move(self, r, c, action):
        dr, dc = 0, 0
        if action == 'up':
            dr = -1
        elif action == 'down':
            dr = 1
        elif action == 'left':
            dc = -1
        elif action == 'right':
            dc = 1

        nr, nc = r + dr, c + dc
        if nr < 0 or nr >= self.nrows or nc < 0 or nc >= self.ncols:
            return r, c
        if self.board[nr][nc] == '#':
            return r, c
        return nr, nc

Teniendo en cuenta la definición del agente, genere un ambiente de `10x10` como se muestra a continuación.

![evaluacion.png](./img/evaluacion.png)


In [54]:
Board=[
['S',' ',' ',' ',' ',' ',' ',' ',' ',' '],
[' ',' ',' ',' ',' ',' ',' ',' ',' ',' '],
[' ','#','#','#','#',' ','#','#','#',' '],
[' ',' ',' ',' ','#',' ',' ',' ',' ',' '],
[' ',' ',' ',' ','#','-1',' ',' ',' ',' '],
[' ',' ',' ',' ','#','+1',' ',' ',' ',' '],
[' ',' ',' ',' ','#',' ',' ',' ',' ',' '],
[' ',' ',' ',' ','#','-1','-1',' ',' ',' '],
[' ',' ',' ',' ',' ',' ',' ',' ',' ',' '],
[' ',' ',' ',' ',' ',' ',' ',' ',' ',' ']
]

In [55]:
def prob(i,j,a):
    if a == 'up':
        if i>0 and Board[i-1][j]!='#':
            return 1
        else:
            return 0
    elif a == 'down':
        if i<len(Board)-1 and Board[i+1][j]!='#':
            return 1
        else:
            return 0
    elif a == 'left':
        if j>0 and Board[i][j-1]!='#':
            return 1
        else:
            return 0
    elif a == 'right':
        if j<len(Board[0])-1 and Board[i][j+1]!='#':
            return 1
        else:
            return 0
    return 0 # for 'exit' action

In [56]:
P=[[[] for _ in range(len(Board[0]))] for _ in range(len(Board))]
for a in ['up', 'down', 'left', 'right', 'exit']:
    for i in range(len(Board)):
        for j in range(len(Board[0])):
            if Board[i][j] in [' ', 'S']:
                P[i][j].append(prob(i,j,a))
            elif Board[i][j] in ['+1', '-1'] and a == 'exit':
                P[i][j].append(1.0)
            elif Board[i][j] in ['+1', '-1'] and a != 'exit':
                P[i][j].append(0)
            elif Board[i][j] == '#':
                P[i][j].append(1)

for i in range(len(P)):
    for j in range(len(P[0])):
        
        suma=sum(P[i][j])
        for k in range(len(P[i][j])):
            p = P[i][j][k]/suma
            P[i][j][k] = p


In [57]:
Env=Environment(board=Board,
             P=P
            )



### Task 2.
Plantee el problema de MDP para cada una de las casillas. Especifique el estado de inicio, las transiciones y su probabilidad (suponiendo que todas las acciones sucede con probabilidad de 0.25) y los estados de fin con su recompensa.
¿Cómo serían las recompensas esperadas para cada estado?



Cada celda (i,j) una acción $k$ tiene la probabilidad indicada en P[i][j][k], 

In [58]:
R=[[0 for _ in range(len(Board[0]))] for _ in range(len(Board))]
for i in range(len(Board)):
    for j in range(len(Board[0])):
        if Board[i][j] == '+1':
            R[i][j] = 1
        elif Board[i][j] == '-1':
            R[i][j] = -1
        else:
            R[i][j] = 0
R

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, -1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, -1, -1, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


### Task 3.
Bajo la definción del problema anterior, suponga que cada acción tiene una probabilidad de éxito de 60%, con probabilidad de 20% se ejecutará la sigiente acción (en dirección de las manesillas del reloj), con probabilidad de 10% se ejecutará la sigiente acción (en contra de las manesillas del reloj) y con probabilidad de 10% no pasará nada. Bajo estas condiciones, ¿Cómo serían las recompensas esperadas para cada estado? 

Codifique el ambiente para el gridworld de `10x10` utilizando esta función de probabilidad. En esta codificación del ambiente no es necesario pasar la matriz `P` como parámetro, pero esta información se debe tener en cuenta en la función `do_action`.

Tenga en cuenta que la calidad del programa que entreguen será tenida en cuentra dentro de la calificación.



En este caso, $P$ cambia a que cada celda $(i,j,k, k' )$ una acción $k$ contiene las probabilidades de que intentando hacer $k$ se haga $k'$. Y $P$ se calcula directamente en el enviroment.

In [69]:
import random

class EnvironmentNuevo:
    def __init__(self, board):
        self.board = board
        self.nrows = len(board)
        self.ncols = len(board[0]) 
        self.initial_state = self._find_initial_state()
        self.current_state = self.initial_state
        self.actions = ['up', 'right', 'down', 'left', 'exit']
        self.P = self._build_transition_matrix()
        
    def _build_transition_matrix(self):
        nA = len(self.actions)
    
        P = [[[[0 for _ in range(nA)] for _ in range(nA)]
              for _ in range(self.ncols)]
              for _ in range(self.nrows)]
    
        for i in range(self.nrows):
            for j in range(self.ncols):
            
                if self.board[i][j] == '#':
                    continue
                
                # Si es estado terminal
                if self._is_exit(i, j):
                    P[i][j][4][4] = 1.0  # exit → exit
                    continue
                
                for a in range(4):  # solo up,right,down,left
                
                    clockwise = (a + 1) % 4
                    counterclock = (a - 1) % 4
    
                    P[i][j][a][a] = 0.6
                    P[i][j][a][clockwise] = 0.2
                    P[i][j][a][counterclock] = 0.1
                    P[i][j][a][a] += 0.1  # 10% quedarse = misma acción
    
        return P


    def _find_initial_state(self):
        for i in range(self.nrows):
            for j in range(self.ncols):
                if self.board[i][j] == 'S':
                    return (i, j)
        return (0, 0)

    def get_current_state(self):
        return self.current_state

    def get_posible_actions(self, state):
        r, c = state
        if self._is_exit(r, c):
            return ['exit']
        return ['up', 'right', 'down', 'left']

    def do_action(self, idx_action):
        r, c = self.current_state

        # Si estamos en estado terminal
        if self._is_exit(r, c):
            if idx_action == 4:  # 'exit' action
                return float(self.board[r][c]), self.current_state
            return 0, self.current_state


        probs = self.P[r][c][idx_action][idx_action]
        if probs ==0:
            return 0, self.current_state


        # Elegimos la acción real según probabilidades
        chosen_action = random.choices(self.actions, weights=self.P[r][c][idx_action], k=1)[0]
        print(f"Chosen action: {chosen_action}, New position: ({r}, {c})")

        nr, nc = self._move(r, c, chosen_action)
        self.current_state = (nr, nc)

        if self._is_exit(nr, nc):
            reward = float(self.board[nr][nc])
        else:
            reward = 0
        return reward, self.current_state


    def reset(self):
        self.current_state = self.initial_state

    def is_terminal(self):
        r, c = self.current_state
        return self._is_exit(r, c)

    def _is_exit(self, r, c):
        val = self.board[r][c]
        return isinstance(val, str) and val.strip() in ['+1', '-1'] or isinstance(val, (int, float))

    def _move(self, r, c, action):
        dr, dc = 0, 0
        if action == 'up':
            dr = -1
        elif action == 'down':
            dr = 1
        elif action == 'left':
            dc = -1
        elif action == 'right':
            dc = 1

        nr, nc = r + dr, c + dc
        if nr < 0 or nr >= self.nrows or nc < 0 or nc >= self.ncols:
            return r, c
        if self.board[nr][nc] == '#':
            return r, c
        return nr, nc

### Pruebas

In [70]:
NuevoEnv = EnvironmentNuevo(Board)
Env

In [71]:
Env.get_posible_actions(Env.get_current_state())

['up', 'down', 'left', 'right']

In [72]:
for action in ['right']*5+['down']*3+['right']+['down']*2+['left']: #Una ruta para salir
    print(f"Current State: {Env.get_current_state()}, Possible Actions: {Env.get_posible_actions(Env.get_current_state())}")
    idx_action = Env.get_posible_actions(Env.get_current_state()).index(action)
    reward, new_state = Env.do_action(idx_action)
    # print(f"Action: {action}, Reward: {reward}, New State: {new_state}")

Current State: (0, 2), Possible Actions: ['up', 'down', 'left', 'right']
Chosen action: left, New position: (0, 1)
Current State: (0, 1), Possible Actions: ['up', 'down', 'left', 'right']
Chosen action: down, New position: (1, 1)
Current State: (1, 1), Possible Actions: ['up', 'down', 'left', 'right']
Chosen action: left, New position: (1, 0)
Current State: (1, 0), Possible Actions: ['up', 'down', 'left', 'right']
Chosen action: up, New position: (0, 0)
Current State: (0, 0), Possible Actions: ['up', 'down', 'left', 'right']
Chosen action: down, New position: (1, 0)
Current State: (1, 0), Possible Actions: ['up', 'down', 'left', 'right']
Chosen action: down, New position: (2, 0)
Current State: (2, 0), Possible Actions: ['up', 'down', 'left', 'right']
Chosen action: down, New position: (3, 0)
Current State: (3, 0), Possible Actions: ['up', 'down', 'left', 'right']
Chosen action: up, New position: (2, 0)
Current State: (2, 0), Possible Actions: ['up', 'down', 'left', 'right']
Current Sta

In [73]:
for action in ['right']*5+['down']*3+['right']+['down']*2+['left']: #Una ruta para salir
    print(f"Current State: {NuevoEnv.get_current_state()}, Possible Actions: {NuevoEnv.get_posible_actions(NuevoEnv.get_current_state())}")
    idx_action = NuevoEnv.get_posible_actions(NuevoEnv.get_current_state()).index(action)
    reward, new_state = NuevoEnv.do_action(idx_action)
    # print(f"Action: {action}, Reward: {reward}, New State: {new_state}")

Current State: (0, 0), Possible Actions: ['up', 'right', 'down', 'left']
Chosen action: right, New position: (0, 0)
Current State: (0, 1), Possible Actions: ['up', 'right', 'down', 'left']
Chosen action: right, New position: (0, 1)
Current State: (0, 2), Possible Actions: ['up', 'right', 'down', 'left']
Chosen action: down, New position: (0, 2)
Current State: (1, 2), Possible Actions: ['up', 'right', 'down', 'left']
Chosen action: right, New position: (1, 2)
Current State: (1, 3), Possible Actions: ['up', 'right', 'down', 'left']
Chosen action: right, New position: (1, 3)
Current State: (1, 4), Possible Actions: ['up', 'right', 'down', 'left']
Chosen action: right, New position: (1, 4)
Current State: (1, 5), Possible Actions: ['up', 'right', 'down', 'left']
Chosen action: left, New position: (1, 5)
Current State: (1, 4), Possible Actions: ['up', 'right', 'down', 'left']
Chosen action: left, New position: (1, 4)
Current State: (1, 3), Possible Actions: ['up', 'right', 'down', 'left']
Ch


### Task 4. 
Defina una situación de la vide real (de su escogencia) como un MDP.

Imaginemos un robot en una bodega de Amazon que debe llevar cajas desde la zona de almacenamiento hasta la zona de envío. El entorno es dinámico, puede haber obstáculos (otros robots o personas) y existe incertidumbre en los movimientos.

Modelamos el problema como un Proceso de Decisión de Markov (MDP):

$$
\mathcal{M} = (S, A, P, R)
$$

## Estados \($S$\)

$$
S = \{ (x,y,c) \mid x,y \text{ posiciones en la bodega},\; c \in \{0,1\} \}
$$

donde:

- \((x,y)\) representa la posición del robot.
- \(c=0\) indica que el robot no lleva caja.
- \(c=1\) indica que el robot lleva una caja.

Existe un estado terminal cuando la caja ha sido entregada correctamente en la zona de envío.

---

## Acciones \($A$\)

$$
A = \{ \text{Norte, Sur, Este, Oeste, Recoger, Soltar} \}
$$

- **Norte, Sur, Este, Oeste**: movimientos dentro de la bodega.
- **Recoger**: toma una caja si está en zona de almacenamiento.
- **Soltar**: deja la caja si está en la zona de envío.

---
## Transiciones \($P(s' \mid s,a)$\)

Las transiciones son probabilísticas.

### Movimientos:

$$
P(s' \mid s,a) =
\begin{cases}
0.8 & \text{si el movimiento es exitoso} \\
0.1 & \text{si ocurre desviación lateral} \\
0.1 & \text{si permanece en la misma posición}
\end{cases}
$$

### Acción **Recoger**:

$$
P((x,y,1) \mid (x,y,0), \text{Recoger}) =
\begin{cases}
0.9 & \text{si está en zona de almacenamiento} \\
0.1 & \text{si falla la acción}
\end{cases}
$$

En cualquier otro caso, el estado no cambia.

### Acción **Soltar**:

$$
P(\text{terminal} \mid (x,y,1), \text{Soltar}) =
\begin{cases}
1 & \text{si está en zona de envío}
\end{cases}
$$

---

## Función de recompensa \($R(s,a,s')$\)

$$
R(s,a,s') =
\begin{cases}
+100 & \text{si entrega correctamente la caja} \\
-1 & \text{por cada movimiento (costo energético)} \\
-5 & \text{si intenta una acción inválida} \\
0 & \text{en otro caso}
\end{cases}
$$

